In [3]:
!pip install trafilatura

  Using cached trafilatura-2.0.0-py3-none-any.whl.metadata (12 kB)
     ---------------------------------------- 0.0/41.4 kB ? eta -:--:--
     ------------------ ------------------- 20.5/41.4 kB 320.0 kB/s eta 0:00:01
     -------------------------------------- 41.4/41.4 kB 398.9 kB/s eta 0:00:00
  Using cached courlan-1.3.2-py3-none-any.whl.metadata (17 kB)
  Using cached htmldate-1.9.4-py3-none-any.whl.metadata (10 kB)
  Using cached justext-3.0.2-py2.py3-none-any.whl.metadata (7.3 kB)
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB ? eta 0:00:00
  Using cached tzlocal-5.3.1-py3-none-any.whl.metadata (7.6 kB)
Using cached trafilatura-2.0.0-py3-none-any.whl (132 kB)
   ---------------------------------------- 0.0/154.3 kB ? eta -:--:--
   ---------------------------------------- 154.3/154.3 kB ? eta 0:00:00
Using cached courlan-1.3.2-py3-none-any.whl (33 kB)
Using cached htmldate-1.9.4-py3-none-any.whl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.32.0 requires protobuf<5,>=3.20, but you have protobuf 6.33.1 which is incompatible.


In [7]:
!pip install spacy

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-0.4.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached confection-0.1.5-py3-none-any.whl.metadata (19 kB)
  Using cached cloudpathlib-0.23.0-py3-none-any.whl.metadata (16 kB)
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   --- ------------------------------------ 1.1/14.2 MB 23.8 MB/s eta 0:00:01
   ---------- ----------------------------- 3.8/14.2 MB 49.1 MB/s eta 0:00:01
   ------------------------ --------------- 8.8/14.2 MB 70.7 MB/s eta 0:00:01
   -------------------------------------- - 13.6/14.2 MB 110.0 MB/s eta 0:00:01
   ---------------------------------------- 14.2/14.2 MB 92.9 MB/s eta 0:00:00
Using cached catalogue-2.0.10-py3-none-any.whl (17 kB)
   ----------

In [1]:
import trafilatura
import httpx
import json
import spacy
import pandas as pd

# =============================
# CONFIGURATION
# =============================

SEED_URLS = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Knowledge_graph"
]

MIN_WORDS = 500
OUTPUT_JSONL = "crawler_output.jsonl"
OUTPUT_CSV = "extracted_knowledge.csv"

# Load spaCy transformer model
nlp = spacy.load("en_core_web_trf")

# =============================
# PHASE 1 — CRAWLING & CLEANING
# =============================

def fetch_and_clean(url):
    """
    Fetch a web page and extract its main textual content.
    Pages with insufficient content are discarded.
    """
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return None

        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=False
        )

        if not text:
            return None

        word_count = len(text.split())
        if word_count < MIN_WORDS:
            return None

        return {
            "url": url,
            "text": text,
            "word_count": word_count
        }

    except Exception as e:
        print(f"Error processing {url}: {e}")
        return None


print("🔎 Crawling and cleaning pages...")
documents = []

for url in SEED_URLS:
    result = fetch_and_clean(url)
    if result:
        documents.append(result)

# Save cleaned pages to JSONL
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

print(f"✅ Phase 1 complete — {len(documents)} documents saved to {OUTPUT_JSONL}")

# =============================
# PHASE 2 — INFORMATION EXTRACTION
# =============================

entities_data = []
relations_data = []

print("🧠 Extracting entities and relations...")

for doc in documents:
    spacy_doc = nlp(doc["text"])

    # --- Named Entity Recognition ---
    for ent in spacy_doc.ents:
        if ent.label_ in {"PERSON", "ORG", "GPE", "DATE"}:
            entities_data.append({
                "entity": ent.text,
                "type": ent.label_,
                "source_url": doc["url"]
            })

    # --- Simple Relation Extraction ---
    for sent in spacy_doc.sents:
        sent_ents = list(sent.ents)
        if len(sent_ents) >= 2:
            for token in sent:
                if token.pos_ == "VERB":
                    relations_data.append({
                        "subject": sent_ents[0].text,
                        "relation": token.lemma_,
                        "object": sent_ents[1].text,
                        "source_url": doc["url"]
                    })
                    break

# =============================
# SAVE RESULTS
# =============================

entities_df = pd.DataFrame(entities_data).drop_duplicates()
entities_df.to_csv(OUTPUT_CSV, index=False)

print(f"📄 Phase 2 complete — entities saved to {OUTPUT_CSV}")
print(f"🔗 Relations extracted: {len(relations_data)}")

print("\nSample of extracted entities:")
print(entities_df.head())


🔎 Crawling and cleaning pages...
✅ Phase 1 complete — 3 documents saved to crawler_output.jsonl
🧠 Extracting entities and relations...
📄 Phase 2 complete — entities saved to extracted_knowledge.csv
🔗 Relations extracted: 335

Sample of extracted entities:
    entity type                                         source_url
0   Google  ORG  https://en.wikipedia.org/wiki/Artificial_intel...
1  YouTube  ORG  https://en.wikipedia.org/wiki/Artificial_intel...
2   Amazon  ORG  https://en.wikipedia.org/wiki/Artificial_intel...
3  Netflix  ORG  https://en.wikipedia.org/wiki/Artificial_intel...
5   OpenAI  ORG  https://en.wikipedia.org/wiki/Artificial_intel...
